# PropertyLens Data Preparation

目标：将原始公开数据加工为可训练的数据集，覆盖以下特征：

- 房屋楼层（`level_mid`）
- 租约剩余年限（`lease_years_left`）
- 房屋面积（`floor_area_sqm`）
- 房间数量（`room_count`）
- 距离 MRT / 食阁 / 大型商业体
- 1 公里内学校数量与学校档次

数据源（可扩展）：

- data.gov.sg（HDB resale）
- Google Maps Places API（MRT / foodcourt / malls）
- sgschooling（2024 报名数据）

> 说明：本 notebook 默认把 API key 放到环境变量。未配置时会跳过对应步骤并保留占位字段。

In [ ]:
import os
import re
import time
import json
from io import StringIO
from pathlib import Path
from dataclasses import dataclass
from typing import Optional

import numpy as np
import pandas as pd
import requests
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 120)

In [20]:
@dataclass
class Config:
    project_root: Path = Path.cwd()
    raw_dir: Path = Path.cwd() / "data" / "raw"
    interim_dir: Path = Path.cwd() / "data" / "interim"
    processed_dir: Path = Path.cwd() / "data" / "processed"

    hdb_resource_id: str = "f1765b54-a209-4718-8d38-a39237f502b3"
    ckan_base: str = "https://data.gov.sg/api/action"

    google_maps_api_key: Optional[str] = os.getenv("GOOGLE_MAPS_API_KEY")
    sgschooling_base_url: str = "https://sgschooling.com/year/{year}/all"
    school_start_year: int = 2016
    school_end_year: int = pd.Timestamp.today().year

    max_hdb_rows: Optional[int] = 20000  # Demo 上限；设为 None 表示全量
    sample_rows_for_places_api: int = 5000  # 提高覆盖率，避免上下文特征过稀
    places_search_radius_m: int = 3000


cfg = Config()
for p in [cfg.raw_dir, cfg.interim_dir, cfg.processed_dir]:
    p.mkdir(parents=True, exist_ok=True)

cfg

Config(project_root=PosixPath('/Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens'), raw_dir=PosixPath('/Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/data/raw'), interim_dir=PosixPath('/Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/data/interim'), processed_dir=PosixPath('/Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/data/processed'), hdb_resource_id='f1765b54-a209-4718-8d38-a39237f502b3', ckan_base='https://data.gov.sg/api/action', google_maps_api_key=None, sgschooling_base_url='https://sgschooling.com/year/{year}/all', school_start_year=2016, school_end_year=2026, max_hdb_rows=20000, sample_rows_for_places_api=5000, places_search_radius_m=3000)

In [3]:
import shutil


def safe_name(s: str) -> str:
    return re.sub(r"[^a-zA-Z0-9._-]+", "_", s).strip("_").lower()


raw_hdb_dir = cfg.raw_dir / "hdb_resale"
raw_school_dir = cfg.raw_dir / "school"
raw_amenity_dir = cfg.raw_dir / "amenities"
for d in [raw_hdb_dir, raw_school_dir, raw_amenity_dir]:
    d.mkdir(parents=True, exist_ok=True)

# 1) 复制你本地已下载的 HDB 历史文件
legacy_hdb_dir = cfg.project_root / "ResaleFlatPrices"
copied = []
if legacy_hdb_dir.exists():
    for f in legacy_hdb_dir.glob("*.csv"):
        target = raw_hdb_dir / f.name
        shutil.copy2(f, target)
        copied.append(target.name)

print(f"Copied HDB files: {len(copied)}")

# 2) 按年份下载 sgschooling（2016 起）并加 year 维度
school_manifest = []
year_tables = []

for year in range(cfg.school_start_year, cfg.school_end_year + 1):
    url = cfg.sgschooling_base_url.format(year=year)
    html_path = raw_school_dir / f"sgschooling_{year}.html"
    csv_path = raw_school_dir / f"sgschooling_{year}_tables.csv"
    try:
        resp = requests.get(url, timeout=60)
        if resp.status_code != 200:
            school_manifest.append({"year": year, "url": url, "status": f"http_{resp.status_code}"})
            continue

        html_path.write_text(resp.text, encoding="utf-8")
        tables = pd.read_html(StringIO(resp.text))

        if tables:
            stacked = []
            for i, t in enumerate(tables, start=1):
                tmp = t.copy()
                tmp.insert(0, "year", year)
                tmp.insert(1, "table_index", i)
                stacked.append(tmp)
            year_df = pd.concat(stacked, ignore_index=True)
            year_df.to_csv(csv_path, index=False)
            year_tables.append(year_df)
            school_manifest.append({"year": year, "url": url, "status": "ok", "rows": len(year_df)})
            print(f"Saved: {html_path}")
            print(f"Saved: {csv_path}")
        else:
            school_manifest.append({"year": year, "url": url, "status": "no_table"})
    except Exception as e:
        msg = str(e).replace("\n", " ")[:240]
        school_manifest.append({"year": year, "url": url, "status": f"failed: {msg}"})
    time.sleep(0.2)

school_manifest_df = pd.DataFrame(school_manifest)
school_manifest_path = raw_school_dir / "sgschooling_yearly_manifest.csv"
school_manifest_df.to_csv(school_manifest_path, index=False)
print(f"Saved: {school_manifest_path}")

if year_tables:
    school_all_years = pd.concat(year_tables, ignore_index=True)
    school_all_years_path = raw_school_dir / "sgschooling_2016_onwards_tables.csv"
    school_all_years.to_csv(school_all_years_path, index=False)
    print(f"Saved: {school_all_years_path}")

# 3) 下载 OSM 设施点（替代 Google 批量抓取，离线可复用）
def overpass_fetch(query: str) -> dict:
    url = "https://overpass-api.de/api/interpreter"
    r = requests.post(url, data={"data": query}, timeout=180)
    r.raise_for_status()
    return r.json()


def overpass_to_df(payload: dict, amenity_type: str) -> pd.DataFrame:
    rows = []
    for e in payload.get("elements", []):
        lat = e.get("lat")
        lon = e.get("lon")
        tags = e.get("tags", {})
        rows.append(
            {
                "osm_id": e.get("id"),
                "osm_type": e.get("type"),
                "name": tags.get("name"),
                "amenity_type": amenity_type,
                "lat": lat,
                "lon": lon,
                "tags": json.dumps(tags, ensure_ascii=True),
            }
        )
    return pd.DataFrame(rows)


# 新加坡行政范围内抓取
osm_queries = {
    "mrt": """
[out:json][timeout:120];
area["name"="Singapore"]["boundary"="administrative"]->.a;
(
  node["railway"="station"](area.a);
  node["station"="subway"](area.a);
);
out body;
""",
    "hawker": """
[out:json][timeout:120];
area["name"="Singapore"]["boundary"="administrative"]->.a;
(
  node["amenity"="food_court"](area.a);
  node["amenity"="marketplace"](area.a);
);
out body;
""",
    "mall": """
[out:json][timeout:120];
area["name"="Singapore"]["boundary"="administrative"]->.a;
(
  node["shop"="mall"](area.a);
);
out body;
""",
    "school": """
[out:json][timeout:120];
area["name"="Singapore"]["boundary"="administrative"]->.a;
(
  node["amenity"="school"](area.a);
);
out body;
""",
}

manifest_rows = []
for k, q in osm_queries.items():
    try:
        payload = overpass_fetch(q)
        json_path = raw_amenity_dir / f"osm_{k}.json"
        csv_path = raw_amenity_dir / f"osm_{k}.csv"

        json_path.write_text(json.dumps(payload, ensure_ascii=True), encoding="utf-8")
        overpass_to_df(payload, amenity_type=k).to_csv(csv_path, index=False)

        manifest_rows.append({"dataset": k, "json_path": str(json_path), "csv_path": str(csv_path), "status": "ok"})
        print(f"Saved: {json_path}")
        print(f"Saved: {csv_path}")
        time.sleep(1.0)
    except Exception as e:
        manifest_rows.append({"dataset": k, "status": f"failed: {e}"})
        print(f"OSM download failed for {k}: {e}")

manifest_path = cfg.raw_dir / "download_manifest.csv"
pd.DataFrame(manifest_rows).to_csv(manifest_path, index=False)
print(f"Saved: {manifest_path}")
pd.DataFrame(manifest_rows)

Copied HDB files: 5
Saved: /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/data/raw/school/sgschooling_2016.html
Saved: /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/data/raw/school/sgschooling_2016_tables.csv
Saved: /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/data/raw/school/sgschooling_2017.html
Saved: /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/data/raw/school/sgschooling_2017_tables.csv
Saved: /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/data/raw/school/sgschooling_2018.html
Saved: /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/data/raw/school/sgschooling_2018_tables.csv
Saved: /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/data/raw/school/sgschooling_2019.html
Saved: /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/

,dataset,json_path,csv_path,status
0,mrt,/Users/lorenzolou/Library/Mobile Documents/com...,/Users/lorenzolou/Library/Mobile Documents/com...,ok
1,hawker,/Users/lorenzolou/Library/Mobile Documents/com...,/Users/lorenzolou/Library/Mobile Documents/com...,ok
2,mall,/Users/lorenzolou/Library/Mobile Documents/com...,/Users/lorenzolou/Library/Mobile Documents/com...,ok
3,school,NaN,NaN,failed: 429 Client Error: Too Many Requests fo...


In [4]:
# Retry: school layer only (备用 Overpass 实例)
retry_query_school = """
[out:json][timeout:180];
area[\"name\"=\"Singapore\"][\"boundary\"=\"administrative\"]->.a;
(
  node[\"amenity\"=\"school\"](area.a);
);
out body;
"""

retry_endpoints = [
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass-api.de/api/interpreter",
]

school_ok = False
last_err = None
for ep in retry_endpoints:
    try:
        r = requests.post(ep, data={"data": retry_query_school}, timeout=240)
        r.raise_for_status()
        payload = r.json()

        school_json = cfg.raw_dir / "amenities" / "osm_school.json"
        school_csv = cfg.raw_dir / "amenities" / "osm_school.csv"
        school_json.write_text(json.dumps(payload, ensure_ascii=True), encoding="utf-8")
        overpass_to_df(payload, amenity_type="school").to_csv(school_csv, index=False)

        print(f"Saved: {school_json}")
        print(f"Saved: {school_csv}")
        school_ok = True
        break
    except Exception as e:
        last_err = e

if not school_ok:
    print(f"School retry failed: {last_err}")

Saved: /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/data/raw/amenities/osm_school.json
Saved: /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/data/raw/amenities/osm_school.csv


## 下载并缓存其他所需原始数据

这一节会把你已下载的 HDB 历史 CSV 复制到 `data/raw/hdb_resale/`，并自动下载以下数据到本地：

- `sgschooling` 2024 页面快照与可解析表格
- data.gov.sg 上与 `school / mrt / hawker / mall` 相关的 CSV 资源（自动筛选）

这样后续特征工程可以优先读取本地文件，减少重复联网请求。

In [5]:
def ckan_datastore_search(resource_id: str, limit: int = 1000, offset: int = 0) -> dict:
    url = f"{cfg.ckan_base}/datastore_search"
    params = {"resource_id": resource_id, "limit": limit, "offset": offset}
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    return r.json()


def fetch_hdb_resale(max_rows: Optional[int] = None) -> pd.DataFrame:
    frames = []
    offset = 0
    page_size = 5000

    while True:
        payload = ckan_datastore_search(cfg.hdb_resource_id, limit=page_size, offset=offset)
        records = payload.get("result", {}).get("records", [])
        if not records:
            break

        frames.append(pd.DataFrame(records))
        offset += len(records)

        if max_rows is not None and offset >= max_rows:
            break

        # 对公开 API 做节流，避免过快请求
        time.sleep(0.2)

    if not frames:
        return pd.DataFrame()

    df = pd.concat(frames, ignore_index=True)
    if max_rows is not None:
        df = df.head(max_rows).copy()
    return df

In [21]:
def load_hdb_from_local_folder(folder: Path) -> pd.DataFrame:
    parts = []
    for f in sorted(folder.glob("*.csv")):
        try:
            parts.append(pd.read_csv(f))
        except Exception as e:
            print(f"skip {f.name}: {e}")
    if not parts:
        return pd.DataFrame()
    return pd.concat(parts, ignore_index=True)


try:
    hdb_df = fetch_hdb_resale(max_rows=cfg.max_hdb_rows)
    source_mode = "data.gov.sg API"
except Exception as e:
    print(f"API fetch failed, fallback to local CSVs: {e}")
    hdb_df = load_hdb_from_local_folder(cfg.raw_dir / "hdb_resale")
    source_mode = "local hdb_resale CSVs"

raw_hdb_path = cfg.raw_dir / "hdb_resale_raw.csv"
hdb_df.to_csv(raw_hdb_path, index=False)

print("source:", source_mode)
print(f"HDB rows: {len(hdb_df):,}")
print(f"Saved: {raw_hdb_path}")
hdb_df.head(3)

API fetch failed, fallback to local CSVs: 429 Client Error: Too Many Requests for url: https://data.gov.sg/api/action/datastore_search?resource_id=f1765b54-a209-4718-8d38-a39237f502b3&limit=5000&offset=0
source: local hdb_resale CSVs
HDB rows: 972,474
Saved: /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/data/raw/hdb_resale_raw.csv


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease
0,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,10 TO 12,31.0,IMPROVED,1977,9000.0,NaN
1,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,04 TO 06,31.0,IMPROVED,1977,6000.0,NaN
2,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,10 TO 12,31.0,IMPROVED,1977,8000.0,NaN


In [22]:
def parse_storey_mid(storey_range: str) -> float:
    if not isinstance(storey_range, str) or " TO " not in storey_range:
        return np.nan
    low, high = storey_range.split(" TO ")
    return (float(low) + float(high)) / 2.0


def parse_room_count(flat_type: str) -> float:
    # 例如: "3 ROOM", "4 ROOM", "EXECUTIVE"
    if not isinstance(flat_type, str):
        return np.nan
    m = re.search(r"(\d+)", flat_type)
    return float(m.group(1)) if m else np.nan


def to_year_month(s: str) -> pd.Timestamp:
    try:
        return pd.to_datetime(s + "-01")
    except Exception:
        return pd.NaT


def add_structural_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["month_dt"] = out["month"].astype(str).map(to_year_month)
    out["lease_commence_date"] = pd.to_numeric(out["lease_commence_date"], errors="coerce")
    out["level_mid"] = out["storey_range"].map(parse_storey_mid)
    out["room_count"] = out["flat_type"].map(parse_room_count)

    out["lease_expiry_year"] = out["lease_commence_date"] + 99
    out["tx_year"] = out["month_dt"].dt.year
    out["lease_years_left"] = out["lease_expiry_year"] - out["tx_year"]

    out["resale_price"] = pd.to_numeric(out["resale_price"], errors="coerce")
    out["floor_area_sqm"] = pd.to_numeric(out["floor_area_sqm"], errors="coerce")

    # OneMap 对不带 ", Singapore" 的命中率更高
    out["address"] = out["block"].astype(str).str.strip() + " " + out["street_name"].astype(str).str.strip()
    return out


hdb_feat = add_structural_features(hdb_df)
hdb_feat.head(3)

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease,month_dt,level_mid,room_count,lease_expiry_year,tx_year,lease_years_left,address
0,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,10 TO 12,31.0,IMPROVED,1977,9000.0,NaN,1990-01-01,11.0,1.0,2076,1990,86,309 ANG MO KIO AVE 1
1,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,04 TO 06,31.0,IMPROVED,1977,6000.0,NaN,1990-01-01,5.0,1.0,2076,1990,86,309 ANG MO KIO AVE 1
2,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,10 TO 12,31.0,IMPROVED,1977,8000.0,NaN,1990-01-01,11.0,1.0,2076,1990,86,309 ANG MO KIO AVE 1


In [ ]:
def onemap_search(query: str) -> tuple[float, float]:
    url = "https://www.onemap.gov.sg/api/common/elastic/search"
    candidates = [query, query.replace(", Singapore", ""), query.replace(",", " ")]
    for q in candidates:
        params = {"searchVal": q, "returnGeom": "Y", "getAddrDetails": "Y", "pageNum": 1}
        r = requests.get(url, params=params, timeout=30)
        r.raise_for_status()
        data = r.json()
        results = data.get("results", [])
        if results:
            return (float(results[0]["LATITUDE"]), float(results[0]["LONGITUDE"]))
    return (np.nan, np.nan)


def geocode_addresses(addresses: pd.Series, max_rows: Optional[int] = None) -> pd.DataFrame:
    uniq = pd.Series(addresses.dropna().unique(), name="address")
    if max_rows is not None:
        uniq = uniq.head(max_rows)

    rows = []
    for addr in uniq:
        try:
            lat, lon = onemap_search(addr)
        except Exception:
            lat, lon = (np.nan, np.nan)
        rows.append({"address": addr, "lat": lat, "lon": lon})
        time.sleep(0.03)

    return pd.DataFrame(rows)


geo_cache_path = cfg.interim_dir / "address_geocode.csv"
address_target = min(cfg.sample_rows_for_places_api, hdb_feat["address"].nunique())
need_refresh = True
if geo_cache_path.exists():
    geocode_df = pd.read_csv(geo_cache_path)
    non_null = int(geocode_df["lat"].notna().sum()) if "lat" in geocode_df.columns else 0
    need_refresh = len(geocode_df) < address_target or non_null < max(100, int(0.7 * address_target))

if need_refresh:
    geocode_df = geocode_addresses(hdb_feat["address"], max_rows=cfg.sample_rows_for_places_api)
    geocode_df.to_csv(geo_cache_path, index=False)

hdb_geo = hdb_feat.merge(geocode_df, on="address", how="left")
print("geocode rows:", len(geocode_df), "non-null:", int(geocode_df["lat"].notna().sum()))
print("geocoded tx rows:", int(hdb_geo["lat"].notna().sum()))
hdb_geo[["address", "lat", "lon"]].head(3)

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0
    p1 = np.radians(lat1)
    p2 = np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2.0) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlambda / 2.0) ** 2
    return 2 * r * np.arctan2(np.sqrt(a), np.sqrt(1 - a))


def google_nearest_place(lat: float, lon: float, keyword: str, place_type: Optional[str] = None) -> float:
    if not cfg.google_maps_api_key or np.isnan(lat) or np.isnan(lon):
        return np.nan
    url = "https://maps.googleapis.com/maps/api/place/nearbysearch/json"
    params = {"location": f"{lat},{lon}", "radius": cfg.places_search_radius_m, "keyword": keyword, "key": cfg.google_maps_api_key}
    if place_type:
        params["type"] = place_type
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    results = r.json().get("results", [])
    if not results:
        return np.nan
    vals = []
    for x in results:
        loc = x.get("geometry", {}).get("location", {})
        plat, plon = loc.get("lat"), loc.get("lng")
        if plat is None or plon is None:
            continue
        vals.append(haversine_km(lat, lon, float(plat), float(plon)))
    return float(np.min(vals)) if vals else np.nan


def google_count_places_within_km(lat: float, lon: float, keyword: str, km: float = 2.0, place_type: Optional[str] = None) -> float:
    if not cfg.google_maps_api_key or np.isnan(lat) or np.isnan(lon):
        return np.nan
    url = "https://maps.googleapis.com/maps/api/place/nearbysearch/json"
    params = {"location": f"{lat},{lon}", "radius": int(km * 1000), "keyword": keyword, "key": cfg.google_maps_api_key}
    if place_type:
        params["type"] = place_type
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return float(len(r.json().get("results", [])))


def _load_osm_points(csv_path: Path) -> np.ndarray:
    if not csv_path.exists():
        return np.empty((0, 2))
    t = pd.read_csv(csv_path)
    if not {"lat", "lon"}.issubset(set(t.columns)):
        return np.empty((0, 2))
    return t[["lat", "lon"]].dropna().astype(float).values


def _nearest_from_points(lat: float, lon: float, points: np.ndarray) -> float:
    if len(points) == 0 or np.isnan(lat) or np.isnan(lon):
        return np.nan
    return float(np.min(haversine_km(lat, lon, points[:, 0], points[:, 1])))


def _count_within_from_points(lat: float, lon: float, points: np.ndarray, km: float) -> float:
    if len(points) == 0 or np.isnan(lat) or np.isnan(lon):
        return np.nan
    return float(np.sum(haversine_km(lat, lon, points[:, 0], points[:, 1]) <= km))


def build_amenity_features(df_geo: pd.DataFrame) -> pd.DataFrame:
    uniq = df_geo[["address", "lat", "lon"]].drop_duplicates().dropna(subset=["lat", "lon"])
    osm_mrt = _load_osm_points(cfg.raw_dir / "amenities" / "osm_mrt.csv")
    osm_hawker = _load_osm_points(cfg.raw_dir / "amenities" / "osm_hawker.csv")
    osm_mall = _load_osm_points(cfg.raw_dir / "amenities" / "osm_mall.csv")

    rows = []
    for _, rec in uniq.iterrows():
        lat, lon = float(rec["lat"]), float(rec["lon"])
        if cfg.google_maps_api_key:
            mrt_km = google_nearest_place(lat, lon, keyword="MRT station", place_type="subway_station")
            food_km = google_nearest_place(lat, lon, keyword="hawker centre")
            mall_count_2km = google_count_places_within_km(lat, lon, keyword="shopping mall", km=2.0)
        else:
            mrt_km = _nearest_from_points(lat, lon, osm_mrt)
            food_km = _nearest_from_points(lat, lon, osm_hawker)
            mall_count_2km = _count_within_from_points(lat, lon, osm_mall, km=2.0)

        rows.append({
            "address": rec["address"],
            "dist_mrt_km": mrt_km,
            "dist_foodcourt_km": food_km,
            "num_malls_2km": mall_count_2km,
        })

    out = pd.DataFrame(rows)
    for c in ["address", "dist_mrt_km", "dist_foodcourt_km", "num_malls_2km"]:
        if c not in out.columns:
            out[c] = np.nan
    return out


amenity_cache_path = cfg.interim_dir / "amenity_features.csv"
needed_addresses = int(hdb_geo[["address", "lat", "lon"]].drop_duplicates().dropna(subset=["lat", "lon"]).shape[0])
need_refresh_amenity = True
if amenity_cache_path.exists():
    try:
        amenity_df = pd.read_csv(amenity_cache_path)
        ok_cols = {"address", "dist_mrt_km", "dist_foodcourt_km", "num_malls_2km"}.issubset(set(amenity_df.columns))
        has_rows = len(amenity_df) >= max(100, int(0.7 * needed_addresses))
        need_refresh_amenity = not (ok_cols and has_rows)
    except Exception:
        need_refresh_amenity = True

if need_refresh_amenity:
    amenity_df = build_amenity_features(hdb_geo)
    amenity_df.to_csv(amenity_cache_path, index=False)

print("amenity rows:", len(amenity_df), "needed addresses:", needed_addresses)
amenity_df.head(3)

amenity rows: 295


,address,dist_mrt_km,dist_foodcourt_km,num_malls_2km
0,309 ANG MO KIO AVE 1,1.252222,0.672322,0.0
1,216 ANG MO KIO AVE 1,0.990384,0.199932,0.0
2,211 ANG MO KIO AVE 3,0.878822,0.289012,0.0


In [15]:
def extract_first_number(x) -> float:
    if pd.isna(x):
        return np.nan
    m = re.search(r"(\d+(?:\.\d+)?)", str(x))
    return float(m.group(1)) if m else np.nan


def parse_sgschooling_all_years(local_csv: Path) -> pd.DataFrame:
    if not local_csv.exists():
        return pd.DataFrame()

    df = pd.read_csv(local_csv)
    if "School" not in df.columns or "year" not in df.columns:
        return pd.DataFrame()

    out = df.copy()
    out["School"] = out["School"].astype(str).str.strip()

    # 学校名行不以箭头开头，箭头行是 vacancy/applied/taken
    out["row_type"] = np.where(
        out["School"].str.startswith("↳ Vacancy"),
        "vacancy",
        np.where(
            out["School"].str.startswith("↳ Applied"),
            "applied",
            np.where(out["School"].str.startswith("↳ Taken"), "taken", "school"),
        ),
    )

    out["school_name"] = np.where(out["row_type"] == "school", out["School"], np.nan)
    out["school_name"] = out["school_name"].ffill()

    phase_cols = [c for c in ["Phase 1", "2A", "2B", "2C", "2C(S)", "3", "Total Vacancy"] if c in out.columns]
    for c in phase_cols:
        out[c] = out[c].map(extract_first_number)

    out["value_sum"] = out[phase_cols].sum(axis=1, min_count=1) if phase_cols else np.nan
    out["year"] = pd.to_numeric(out["year"], errors="coerce")

    grp = ["year", "school_name"]
    app = out[out["row_type"] == "applied"].groupby(grp, as_index=False)["value_sum"].sum().rename(columns={"value_sum": "applications_total"})
    vac = out[out["row_type"] == "vacancy"].groupby(grp, as_index=False)["value_sum"].sum().rename(columns={"value_sum": "vacancy_total"})

    school_feat = app.merge(vac, on=grp, how="outer")
    school_feat["demand_supply_ratio"] = school_feat["applications_total"] / school_feat["vacancy_total"].replace(0, np.nan)
    return school_feat


def normalize_school_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "school_name" not in out.columns:
        out["school_name"] = np.nan
    out["school_name"] = out["school_name"].astype(str).str.strip()
    out["year"] = pd.to_numeric(out.get("year"), errors="coerce")
    out["demand_supply_ratio"] = pd.to_numeric(out.get("demand_supply_ratio"), errors="coerce")
    return out


school_multi_year_path = cfg.raw_dir / "school" / "sgschooling_2016_onwards_tables.csv"
school_raw = parse_sgschooling_all_years(school_multi_year_path)
school_feat = normalize_school_features(school_raw)

school_raw_path = cfg.raw_dir / "school_2016_onwards_raw.csv"
school_feat_path = cfg.interim_dir / "school_2016_onwards_features.csv"
school_raw.to_csv(school_raw_path, index=False)
school_feat.to_csv(school_feat_path, index=False)

print(f"school rows: {len(school_feat):,}")
if not school_feat.empty and "year" in school_feat.columns:
    years = sorted(school_feat["year"].dropna().astype(int).unique().tolist())
    print("available years:", years)
school_feat.head(10)

school rows: 1,824
available years: [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


,year,school_name,applications_total,vacancy_total,demand_supply_ratio
0,2016,Admiralty,125.0,152.0,0.822368
1,2016,Ahmad Ibrahim,60.0,318.0,0.188679
2,2016,Ai Tong,92.0,72.0,1.277778
3,2016,Alexandra,150.0,200.0,0.750000
4,2016,Anchor Green,141.0,356.0,0.396067
5,2016,Anderson,99.0,118.0,0.838983
6,2016,Ang Mo Kio,63.0,303.0,0.207921
7,2016,Anglo-Chinese (Junior),144.0,99.0,1.454545
8,2016,Anglo-Chinese (Primary),54.0,41.0,1.317073
9,2016,Angsana,15.0,287.0,0.052265


In [16]:
def school_tier_with_knn_style(df: pd.DataFrame, n_clusters: int = 3) -> pd.DataFrame:
    out = df.copy()

    feature_cols = [c for c in ["demand_supply_ratio"] if c in out.columns]
    if not feature_cols:
        out["school_tier"] = np.nan
        return out

    x = out[feature_cols].replace([np.inf, -np.inf], np.nan)
    x = x.fillna(x.median(numeric_only=True))

    scaler = StandardScaler()
    xs = scaler.fit_transform(x)

    km = KMeans(n_clusters=n_clusters, random_state=42, n_init=20)
    labels = km.fit_predict(xs)
    out["cluster"] = labels

    # 聚类中心竞争度越高，tier 越高
    cluster_rank = (
        out.groupby("cluster")["demand_supply_ratio"]
        .mean()
        .sort_values(ascending=False)
        .reset_index()
    )
    cluster_rank["school_tier"] = [f"tier_{i+1}" for i in range(len(cluster_rank))]

    out = out.merge(cluster_rank[["cluster", "school_tier"]], on="cluster", how="left")
    return out


school_tier_df = school_tier_with_knn_style(school_feat, n_clusters=3)
school_tier_df.head(3)

,year,school_name,applications_total,vacancy_total,demand_supply_ratio,cluster,school_tier
0,2016,Admiralty,125.0,152.0,0.822368,1,tier_2
1,2016,Ahmad Ibrahim,60.0,318.0,0.188679,0,tier_3
2,2016,Ai Tong,92.0,72.0,1.277778,2,tier_1


In [18]:
def geocode_schools(df: pd.DataFrame, max_rows: int = 200) -> pd.DataFrame:
    if "school_name" not in df.columns:
        return pd.DataFrame(columns=["school_name", "school_lat", "school_lon"])

    names = df["school_name"].dropna().astype(str).str.strip().drop_duplicates().head(max_rows)
    rows = []
    for n in names:
        try:
            lat, lon = onemap_search(f"{n}, Singapore")
        except Exception:
            lat, lon = (np.nan, np.nan)
        rows.append({"school_name": n, "school_lat": lat, "school_lon": lon})
        time.sleep(0.15)
    return pd.DataFrame(rows)


def nearest_school_features(home_df: pd.DataFrame, school_df: pd.DataFrame) -> pd.DataFrame:
    homes = home_df[["address", "lat", "lon"]].drop_duplicates().dropna(subset=["lat", "lon"]).copy()
    schools = school_df.dropna(subset=["school_lat", "school_lon"]).copy()

    if homes.empty or schools.empty:
        return pd.DataFrame(columns=["address", "nearest_school_dist_km", "schools_within_1km", "best_tier_within_1km"])

    rows = []
    for _, h in homes.iterrows():
        d = haversine_km(
            float(h["lat"]),
            float(h["lon"]),
            schools["school_lat"].astype(float).values,
            schools["school_lon"].astype(float).values,
        )

        nearest = float(np.min(d))
        within = schools[d <= 1.0].copy()
        cnt_1km = int(len(within))

        if cnt_1km > 0 and "school_tier" in within.columns:
            # tier_1 > tier_2 > tier_3
            ranked = (
                within.assign(tier_num=within["school_tier"].str.extract(r"(\d+)").astype(float))
                .sort_values("tier_num")
            )
            best_tier = ranked["school_tier"].iloc[0]
        else:
            best_tier = np.nan

        rows.append(
            {
                "address": h["address"],
                "nearest_school_dist_km": nearest,
                "schools_within_1km": cnt_1km,
                "best_tier_within_1km": best_tier,
            }
        )

    return pd.DataFrame(rows)


school_geo_cache = cfg.interim_dir / "school_geocode.csv"
if school_geo_cache.exists():
    school_geo_df = pd.read_csv(school_geo_cache)
else:
    school_geo_df = geocode_schools(school_tier_df, max_rows=300)
    school_geo_df.to_csv(school_geo_cache, index=False)

school_joined = school_tier_df.merge(school_geo_df, on="school_name", how="left")
home_school_feat = nearest_school_features(hdb_geo, school_joined)
home_school_feat.head(3)

,address,nearest_school_dist_km,schools_within_1km,best_tier_within_1km
0,309 ANG MO KIO AVE 1,0.137805,50,tier_1
1,216 ANG MO KIO AVE 1,0.498246,30,tier_2
2,211 ANG MO KIO AVE 3,0.420839,50,tier_1


In [19]:
train_df = (
    hdb_geo.merge(amenity_df, on="address", how="left")
    .merge(home_school_feat, on="address", how="left")
)

# 增加年维度（交易年份）
train_df["year"] = train_df["tx_year"]

# 将 tier 文本映射为可训练数值
train_df["best_school_tier_score"] = (
    train_df["best_tier_within_1km"]
    .astype(str)
    .str.extract(r"(\d+)")
    .astype(float)
)
train_df["best_school_tier_score"] = 4 - train_df["best_school_tier_score"]

feature_cols = [
    "level_mid",
    "lease_years_left",
    "floor_area_sqm",
    "room_count",
    "dist_mrt_km",
    "dist_foodcourt_km",
    "num_malls_2km",
    "nearest_school_dist_km",
    "schools_within_1km",
    "best_school_tier_score",
]

model_df = train_df.dropna(subset=["resale_price"]).copy()

# 对演示 notebook 采用中位数填充，生产环境建议按区域分桶填充
for c in feature_cols:
    model_df[c] = pd.to_numeric(model_df[c], errors="coerce")
    model_df[c] = model_df[c].fillna(model_df[c].median())

training_dataset = model_df[["year", "month", "town", "flat_type", "address", "resale_price"] + feature_cols].copy()

processed_path_csv = cfg.processed_dir / "training_dataset.csv"
processed_path_parquet = cfg.processed_dir / "training_dataset.parquet"
training_dataset.to_csv(processed_path_csv, index=False)
training_dataset.to_parquet(processed_path_parquet, index=False)

print(f"training rows: {len(training_dataset):,}")
print(f"saved: {processed_path_csv}")
print(f"saved: {processed_path_parquet}")
training_dataset.head(5)

training rows: 972,474
saved: /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/data/processed/training_dataset.csv
saved: /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/data/processed/training_dataset.parquet


,year,month,town,flat_type,address,resale_price,level_mid,lease_years_left,floor_area_sqm,room_count,dist_mrt_km,dist_foodcourt_km,num_malls_2km,nearest_school_dist_km,schools_within_1km,best_school_tier_score
0,1990,1990-01,ANG MO KIO,1 ROOM,309 ANG MO KIO AVE 1,9000.0,11.0,86,31.0,1.0,1.252222,0.672322,0.0,0.137805,50.0,3.0
1,1990,1990-01,ANG MO KIO,1 ROOM,309 ANG MO KIO AVE 1,6000.0,5.0,86,31.0,1.0,1.252222,0.672322,0.0,0.137805,50.0,3.0
2,1990,1990-01,ANG MO KIO,1 ROOM,309 ANG MO KIO AVE 1,8000.0,11.0,86,31.0,1.0,1.252222,0.672322,0.0,0.137805,50.0,3.0
3,1990,1990-01,ANG MO KIO,1 ROOM,309 ANG MO KIO AVE 1,6000.0,8.0,86,31.0,1.0,1.252222,0.672322,0.0,0.137805,50.0,3.0
4,1990,1990-01,ANG MO KIO,3 ROOM,216 ANG MO KIO AVE 1,47200.0,5.0,85,73.0,3.0,0.990384,0.199932,0.0,0.498246,30.0,2.0


## 输出说明

产物文件：

- `data/raw/hdb_resale/`（你下载的 5 份历史 resale CSV）
- `data/raw/school/sgschooling_2016_onwards_tables.csv`
- `data/raw/school/sgschooling_yearly_manifest.csv`
- `data/interim/address_geocode.csv`
- `data/interim/amenity_features.csv`
- `data/interim/school_2016_onwards_features.csv`
- `data/interim/school_geocode.csv`
- `data/processed/training_dataset.csv`（包含 `year` 字段）
- `data/processed/training_dataset.parquet`（包含 `year` 字段）

建议下一步：

1. 将 `GOOGLE_MAPS_API_KEY` 写入环境变量后重新运行 amenities 相关单元。
2. 把 `max_hdb_rows` 调整为 `None` 做全量训练集。
3. 对学校分层引入更多标签特征（如 phase2C、ballot rate），提升 tier 稳定性。